# Gatys Neural Style Transfer: Follow the PyTorch Tutorial

**Learning goals:**

- Explain content loss and Gram-matrix style loss in Gatys neural style transfer.
- Extract reference features with a frozen VGG19 network.
- Optimise image pixels to combine a photograph's content with a reference style.
- Inspect the generated image and the recorded losses.

**Prerequisites:**

- Session 11: CNN features and image style transfer.
- Python with PyTorch, torchvision, Pillow and Matplotlib installed.
- Run this notebook from `assets/style-transfer/` so its input paths resolve.
- The supplied `inputs/content.jpg` and `inputs/style-udnie.jpg` images.
- Internet access for the first download of VGG19 weights; a GPU is optional.

## 1. Introduction

Gatys style transfer fixes a pretrained CNN and updates a generated image's pixels. It repeats optimisation for each new photograph or style reference. No transformation network is trained.

Content loss preserves spatial features of the photograph. Style loss matches correlations between feature channels. The optimiser receives the image tensor, not CNN parameters.

The diagram's content image p supplies target features P. The style image a supplies target Gram matrices A. The generated image x supplies current content features F and style matrices G. CNN weights stay fixed; only x changes.

This notebook follows the PyTorch example using VGG19 and content and style losses, with Adam for direct image-pixel updates. Content is measured in VGG19 block 4, and style is measured across all five blocks. Layer names count convolutions in their order in VGG19.

- [Gatys et al.: Image Style Transfer Using Convolutional Neural Networks](https://www.cv-foundation.org/openaccess/content_cvpr_2016/papers/Gatys_Image_Style_Transfer_CVPR_2016_paper.pdf)
- [Example: Neural Transfer Using PyTorch](https://docs.pytorch.org/tutorials/advanced/neural_style_tutorial.html)


## 2. Select device and load image

Select CUDA, MPS on a supported Mac, or CPU. Use a fixed seed for repeatable noise initialisation.

Define image_loader, imshow, show_comparison and save_image here. Displaying and saving use tensor_to_image to convert a detached tensor to a PIL image.

Resize both reference images to (imsize, imsize), convert RGB pixels to tensors, add the batch dimension and move them to the selected device. Square resizing gives matching shapes and can change the source aspect ratio. Load and display the references after defining the functions.


In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # Use the Mac GPU when MPS is available.
else:
    device = torch.device("cpu")  # Support Intel Macs and unavailable MPS.

SEED = 42
torch.manual_seed(SEED)  # Make noise initialisation repeatable.
# To force CPU execution, set device = torch.device("cpu") here.
print("Device:", device)


In [ ]:
import os
from PIL import Image, ImageOps
from torchvision import transforms
import matplotlib.pyplot as plt

imsize = 512 if device.type == "cuda" else 256 if device.type == "mps" else 128
# Choose the output height and width before constructing the loader.

loader = transforms.Compose([
    transforms.Resize((imsize, imsize)),  # Give both reference images matching shapes.
    transforms.ToTensor(),  # Convert RGB values from [0, 255] to [0, 1].
])
unloader = transforms.ToPILImage()

def image_loader(image_name):
    with Image.open(image_name) as image:
        image = ImageOps.exif_transpose(image).convert("RGB")
        image = loader(image).unsqueeze(0)  # Add the batch dimension.
    return image.to(device, torch.float)

def tensor_to_image(tensor):
    # Detach the visualisation copy, move it to CPU and remove its batch dimension.
    pixels = tensor.detach().cpu().squeeze(0).clamp(0, 1)
    return unloader(pixels)

def imshow(tensor, title):
    image = tensor_to_image(tensor)
    plt.figure(figsize=(5, 5))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()

def show_comparison(content_image, generated_image):
    # Use the same figure size and conversion for a side-by-side comparison.
    figure, axes = plt.subplots(1, 2, figsize=(10, 5))
    for axis, tensor, title in zip(
        axes, (content_image, generated_image), ("Content image", "Final stylised image")
    ):
        axis.imshow(tensor_to_image(tensor))
        axis.set_title(title)
        axis.axis("off")
    figure.tight_layout()
    plt.show()

def save_image(tensor, filename):
    # Create the destination folder, then save through the shared PIL conversion.
    os.makedirs(os.path.dirname(os.path.abspath(filename)), exist_ok=True)
    tensor_to_image(tensor).save(filename)


In [ ]:
CONTENT_PATH = "inputs/content.jpg"
STYLE_PATH = "inputs/style-udnie.jpg"

content_img = image_loader(CONTENT_PATH)
style_img = image_loader(STYLE_PATH)
imshow(style_img, "Style image a")
imshow(content_img, "Content image p")


## 3. Content loss and style loss

Compare current content features F with the fixed content target P. For style, compare the generated image's Gram matrices G with the reference matrices A and sum the losses across layers.

These functions return scalar losses directly.


In [ ]:
from torch.nn import functional as F

def content_loss(content_F, content_P):
    return sum(F.mse_loss(current, target)
               for current, target in zip(content_F, content_P))

def gram_matrix(feature_map):
    _, channels, height, width = feature_map.shape
    features = feature_map.reshape(channels, height * width)
    # Channel correlations describe style statistics.
    return torch.mm(features, features.t()) / (channels * height * width)

def style_loss(style_G, style_A):
    return sum(F.mse_loss(current, target)
               for current, target in zip(style_G, style_A))


## 4. Style transfer CNN model

Use fixed VGG19 features. One forward pass extracts the current content features F and style matrices G.

Use conv_10 for content: it is the second convolution in VGG19 block 4. A deeper content representation allows more changes to fine edges and colours while preserving the broad structure.

Use conv_1, conv_3, conv_5, conv_9 and conv_13 for style: the first convolution in each of the five blocks. Their Gram matrices describe patterns at different spatial scales.

Collect each selected convolution's features after its ReLU activation.

Calculate the fixed targets P and A once from the reference images. The generated image uses the same extraction function during optimisation.


In [ ]:
from torch import nn
from torchvision.models import vgg19, VGG19_Weights

content_layers_default = ['conv_10']  # VGG19 block 4, convolution 2.
style_layers_default = ['conv_1', 'conv_3', 'conv_5', 'conv_9', 'conv_13']
# Take style features from the first convolution of each VGG19 block.

WEIGHTS_DIRECTORY = "../../../archive/AI_SS/style-transfer/weights"
# Keep downloaded model weights in the workspace archive.
torch.hub.set_dir(WEIGHTS_DIRECTORY)
cnn = vgg19(weights=VGG19_Weights.DEFAULT).features.to(device).eval()
cnn.requires_grad_(False)  # Freeze VGG weights; input gradients remain available.
for layer in cnn:
    if isinstance(layer, nn.ReLU):
        layer.inplace = False  # Preserve convolution features before ReLU.

normalization_mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
normalization_std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

def extract_content_and_style_features(image):
    value = (image - normalization_mean) / normalization_std
    selected_layers = set(content_layers_default + style_layers_default)
    image_features = {}
    convolution_number = 0

    for layer in cnn:
        value = layer(value)
        if isinstance(layer, nn.Conv2d):
            convolution_number += 1
        elif isinstance(layer, nn.ReLU):
            # Collect the convolution features after their ReLU activation.
            layer_name = f"conv_{convolution_number}"
            if layer_name in selected_layers:
                image_features[layer_name] = value
            if len(image_features) == len(selected_layers):
                break  # Stop after collecting all selected activated features.

    content_features = [image_features[name] for name in content_layers_default]
    style_features = [gram_matrix(image_features[name]) for name in style_layers_default]
    return content_features, style_features

with torch.no_grad():
    content_P, _ = extract_content_and_style_features(content_img)
    _, style_A = extract_content_and_style_features(style_img)


## 5. Run

Set LEARNING_RATE, CONTENT_WEIGHT, STYLE_WEIGHT and NUM_STEPS. Start from a copy of the content photograph so its changes are visible from the first update.

Adam updates the image pixels directly in each iteration. A larger learning rate increases the update scale, while a smaller content weight gives the original structure more freedom to change.

Each iteration extracts F and G, compares them with P and A, computes the weighted loss and updates x. Run 10 updates with a learning rate of 0.03. Display the updated image and print the losses every 5 steps.

Save and compare the final result using the functions from Section 2. Reinitialise x when comparing settings.


In [ ]:
from torch import optim

NUM_STEPS = 10  # Number of image-pixel updates.
LEARNING_RATE = 0.03  # Control the scale of each Adam update.
CONTENT_WEIGHT = 1  # Allow more changes to the original content.
STYLE_WEIGHT = 5000000.0  # Emphasise the reference style.

input_img = content_img.clone().requires_grad_(True)
# Alternative: input_img = torch.rand_like(content_img).requires_grad_(True)
optimizer = optim.Adam([input_img], lr=LEARNING_RATE)
loss_history = []

for step in range(1, NUM_STEPS + 1):
    optimizer.zero_grad()
    content_F, style_G = extract_content_and_style_features(input_img)
    content_score = CONTENT_WEIGHT * content_loss(content_F, content_P)
    style_score = STYLE_WEIGHT * style_loss(style_G, style_A)
    loss = content_score + style_score

    loss.backward()  # Calculate gradients for the generated image pixels.
    optimizer.step()  # Change image pixels; VGG parameters remain fixed.
    with torch.no_grad():
        input_img.clamp_(0, 1)  # Keep updated RGB values in their valid range.

    loss_history.append({
        "content": content_score.item(),
        "style": style_score.item(),
        "total": loss.item(),
    })
    if step % 5 == 0:
        print(f"Step {step}: content={content_score.item():.4f}, "
              f"style={style_score.item():.4f}, total={loss.item():.4f}")
        imshow(input_img, f"Generated image after update {step}")


In [ ]:
OUTPUT_PATH = "../../../archive/AI_SS/style-transfer/outputs/gatys.png"
save_image(input_img, OUTPUT_PATH)
show_comparison(content_img, input_img)


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot([item["content"] for item in loss_history], label="Weighted content loss")
plt.plot([item["style"] for item in loss_history], label="Weighted style loss")
plt.xlabel("Loss evaluation index")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()
